In [123]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [124]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [125]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               'Temperature', 'DewPoint', 'v10n', 
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

#y_test

((2732, 19), (2732,), (171, 19), (171,))

In [126]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print(KNeighborsRegressor.__module__)

sklearn.neighbors._regression


In [127]:
knn_base_WM = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=10,
        weights="distance",
        p = 1,
    ))
])


knn_base_WM.fit(X_train, y_train)
y_pred = knn_base_WM.predict(X_test)
print("R² for DOC for log scaled | Baseline KNN Regressor | WM", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline KNN Regressor | WM", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline KNN Regressor | WM 0.4143479809618491
R² for DOC on normal scaled | Baseline KNN Regressor | WM 0.3331118828793449
R² for DOC on normal: 0.3331118828793449
Final metrics on NORMAL scale (after inverse log):
R²   : 0.3331
MSE  : 2.0797
RMSE : 1.4421
MAE  : 0.8855


In [128]:
#Hyper Param tuning
from sklearn.model_selection import KFold
import optuna

In [129]:

kf = KFold(n_splits=3, shuffle=True, random_state=42)

def objective_knn_WM(trial):

    params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 3, 40),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "p": trial.suggest_categorical("p", [1, 2])  # 1=Manhattan, 2=Euclidean
    }

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(**params))
    ])

    r2_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        
        model.fit(X_tr, y_tr)

        preds = model.predict(X_va)
        r2_scores.append(r2_score(y_va, preds))

    return np.mean(r2_scores)

In [130]:


study_knn_WM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_knn_WM.optimize(objective_knn_WM, n_trials=50)


[I 2026-02-09 17:27:53,457] A new study created in memory with name: no-name-cb01b521-67cf-4641-aef0-b51f054d347c
[I 2026-02-09 17:27:53,520] Trial 0 finished with value: 0.46707180037244805 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.46707180037244805.
[I 2026-02-09 17:27:53,573] Trial 1 finished with value: 0.8862156617081349 and parameters: {'n_neighbors': 8, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.8862156617081349.
[I 2026-02-09 17:27:53,620] Trial 2 finished with value: 0.8161133477949044 and parameters: {'n_neighbors': 3, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8862156617081349.
[I 2026-02-09 17:27:53,672] Trial 3 finished with value: 0.8906537277455685 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'p': 1}. Best is trial 3 with value: 0.8906537277455685.
[I 2026-02-09 17:27:53,735] Trial 4 finished with value: 0.7624693587024151 and parameters: {'n_neighbors': 26, 'weights'

In [131]:

print("Best CV R2:", study_knn_WM.best_value)
print("Best params:", study_knn_WM.best_params)


Best CV R2: 0.9427664567664849
Best params: {'n_neighbors': 3, 'weights': 'distance', 'p': 1}


In [132]:
best_params_WM = study_knn_WM.best_params
print(best_params_WM)

{'n_neighbors': 3, 'weights': 'distance', 'p': 1}


In [133]:
best_params = best_params_WM

best_knn_WM = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", KNeighborsRegressor(**best_params))
])


best_knn_WM.fit(X_train, y_train)
y_pred = best_knn_WM.predict(X_test)

print("R² for DOC for log scaled | SVR | WM:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | SVR | WM", r2_score(y_test_real, y_pred_orig))




# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | SVR | WM: 0.3083948996152438
R² for DOC on normal scaled | SVR | WM 0.16565830806562665
R² for DOC on normal: 0.16565830806562665
Final metrics on NORMAL scale (after inverse log):
R²   : 0.1657
MSE  : 2.6019
RMSE : 1.6130
MAE  : 0.9701


In [ ]:
#No meteo

In [134]:
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [135]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [136]:
TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"

input_features_noMeteo = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 
                'B9',
               "B4_minus_B3",
               "B4_div_B3", 
               "ND_B4_B3", 
               
               #'Temperature', 'DewPoint', 'v10n', #'Precipitation(mm)',
               'doy_sin', 'doy_cos', 
               'latitude', 'longitude'
               ]

X_train = doc_train_augmented[input_features_noMeteo]
y_train = doc_train_augmented[TARGET]

X_test = doc_test[input_features_noMeteo]
y_test = doc_test[TARGET]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((2732, 16), (2732,), (171, 16), (171,))

In [137]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print(KNeighborsRegressor.__module__)

sklearn.neighbors._regression


In [138]:
knn_base_WM = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=10,
        weights="distance",
        p = 2
    ))
])


knn_base_WM.fit(X_train, y_train)
y_pred = knn_base_WM.predict(X_test)
print("R² for DOC for log scaled | Baseline KNN Regressor | NM", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | Baseline SVM Regressor | NM", r2_score(y_test_real, y_pred_orig))


# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")



R² for DOC for log scaled | Baseline KNN Regressor | NM 0.4847163593852678
R² for DOC on normal scaled | Baseline SVM Regressor | NM 0.44314306189047215
R² for DOC on normal: 0.44314306189047215
Final metrics on NORMAL scale (after inverse log):
R²   : 0.4431
MSE  : 1.7365
RMSE : 1.3178
MAE  : 0.8499


In [139]:
#Hyper Param tuning
from sklearn.model_selection import KFold
import optuna

In [140]:

kf = KFold(n_splits=3, shuffle=True, random_state=42)

def objective_knn_NM(trial):

    params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 3, 40),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "p": trial.suggest_categorical("p", [1, 2])  # 1=Manhattan, 2=Euclidean
    }

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(**params))
    ])

    r2_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        
        model.fit(X_tr, y_tr)

        preds = model.predict(X_va)
        r2_scores.append(r2_score(y_va, preds))

    return np.mean(r2_scores)

In [141]:
study_knn_NM = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_knn_NM.optimize(objective_knn_NM, n_trials=50, show_progress_bar=True)


[I 2026-02-09 17:28:33,141] A new study created in memory with name: no-name-ee3d38aa-0c01-4b33-84ee-f6ca60f909a4
Best trial: 3. Best value: 0.878994:   8%|▊         | 4/50 [00:00<00:02, 16.77it/s]

[I 2026-02-09 17:28:33,223] Trial 0 finished with value: 0.46655428958970563 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.46655428958970563.
[I 2026-02-09 17:28:33,284] Trial 1 finished with value: 0.8782579305297782 and parameters: {'n_neighbors': 8, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.8782579305297782.
[I 2026-02-09 17:28:33,331] Trial 2 finished with value: 0.8424170455017618 and parameters: {'n_neighbors': 3, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8782579305297782.
[I 2026-02-09 17:28:33,386] Trial 3 finished with value: 0.8789938109277576 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'p': 1}. Best is trial 3 with value: 0.8789938109277576.


Best trial: 3. Best value: 0.878994:  12%|█▏        | 6/50 [00:00<00:02, 16.11it/s]

[I 2026-02-09 17:28:33,451] Trial 4 finished with value: 0.7376053785147145 and parameters: {'n_neighbors': 26, 'weights': 'distance', 'p': 2}. Best is trial 3 with value: 0.8789938109277576.
[I 2026-02-09 17:28:33,516] Trial 5 finished with value: 0.727348614846348 and parameters: {'n_neighbors': 32, 'weights': 'distance', 'p': 1}. Best is trial 3 with value: 0.8789938109277576.
[I 2026-02-09 17:28:33,580] Trial 6 finished with value: 0.42758182779029563 and parameters: {'n_neighbors': 26, 'weights': 'uniform', 'p': 2}. Best is trial 3 with value: 0.8789938109277576.


Best trial: 3. Best value: 0.878994:  20%|██        | 10/50 [00:00<00:02, 15.77it/s]

...

[I 2026-02-09 17:28:35,593] Trial 40 finished with value: 0.7017619896509454 and parameters: {'n_neighbors': 39, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:35,649] Trial 41 finished with value: 0.955373414485286 and parameters: {'n_neighbors': 3, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:35,704] Trial 42 finished with value: 0.955373414485286 and parameters: {'n_neighbors': 3, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:35,758] Trial 43 finished with value: 0.9015953498893327 and parameters: {'n_neighbors': 7, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.


Best trial: 14. Best value: 0.955373:  92%|█████████▏| 46/50 [00:02<00:00, 17.11it/s]

[I 2026-02-09 17:28:35,816] Trial 44 finished with value: 0.9261825251037851 and parameters: {'n_neighbors': 5, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:35,880] Trial 45 finished with value: 0.854881333147345 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'p': 2}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:35,940] Trial 46 finished with value: 0.9015953498893327 and parameters: {'n_neighbors': 7, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.


Best trial: 14. Best value: 0.955373: 100%|██████████| 50/50 [00:03<00:00, 16.64it/s]

[I 2026-02-09 17:28:36,004] Trial 47 finished with value: 0.7651598254260493 and parameters: {'n_neighbors': 24, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:36,091] Trial 48 finished with value: 0.40867205430281334 and parameters: {'n_neighbors': 30, 'weights': 'uniform', 'p': 2}. Best is trial 14 with value: 0.955373414485286.
[I 2026-02-09 17:28:36,145] Trial 49 finished with value: 0.9401906531489418 and parameters: {'n_neighbors': 4, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.955373414485286.


In [142]:
print("Best CV R2:", study_knn_NM.best_value)
print("Best params:", study_knn_NM.best_params)

Best CV R2: 0.955373414485286
Best params: {'n_neighbors': 3, 'weights': 'distance', 'p': 1}


In [143]:
best_params_NM = study_knn_NM.best_params
print(best_params_NM)

{'n_neighbors': 3, 'weights': 'distance', 'p': 1}


In [144]:
best_params = best_params_NM

best_knn_NM = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(**best_params))
])


best_knn_NM.fit(X_train, y_train)
y_pred = best_knn_NM.predict(X_test)

print("R² for DOC for log scaled | SVR:", r2_score(y_test, y_pred))

y_pred_orig = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal scaled | SVR:", r2_score(y_test_real, y_pred_orig))




# Back-transform predictions and true values
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)
print("R² for DOC on normal:", r2_score(y_test_real, y_pred_real))

# ---- Metrics on normal scale ----
mse  = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test_real, y_pred_real)
r2   = r2_score(y_test_real, y_pred_real)

print("Final metrics on NORMAL scale (after inverse log):")
print(f"R²   : {r2:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

R² for DOC for log scaled | SVR: 0.3457373648152636
R² for DOC on normal scaled | SVR: 0.2556765748575156
R² for DOC on normal: 0.2556765748575156
Final metrics on NORMAL scale (after inverse log):
R²   : 0.2557
MSE  : 2.3211
RMSE : 1.5235
MAE  : 0.9346
